idea from = https://www.kaggle.com/competitions/rogii-wellbore-geology-prediction/discussion/699853 

coding = gemini , clude + me

CV = 15.20 LB 15.585

any suggestion pls put in comment section

In [ ]:
%%writefile config.py

# config.py
import torch

class Config:
    # Paths
    TRAIN_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction/train" 
    TEST_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction/test"
    TYPEWELL_SUFFIX = "__typewell.csv"
    HORIZONTAL_SUFFIX = "__horizontal_well.csv"
    
    # Model Configurations
    BATCH_SIZE = 4
    EPOCHS = 4
    LR = 1e-3
    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
    N_FOLDS = 4
    
    # GeoSteerNet / Dataset Specifics
    NN_SCALE = 4
    T_H = 128  # Typewell history points
    T_F = 128  # Typewell future points
    H_S = 64 // NN_SCALE   # Horizontal step size = 16
    H_H = 16 * NN_SCALE    # Horizontal history = 64
    H_F = 768 - H_H        # Horizontal future = 704
    H_GR_FILTER = 50       # Savgol filter window for Horizontal GR

    TVT_MIN = 8500.0
    TVT_MAX = 13500.0

    # Training Specifics
    DEFAULT_T_STEP = 0.5
    DEFAULT_H_STEP = 64
    ORIG_PAD_LEN = 16384

In [ ]:
%%writefile dataset.py

# dataset.py
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from scipy.signal import savgol_filter
import cv2
from pathlib import Path
from config import Config

def resample_typewell_by_step(t, target_step=0.5):
    t_tvt = t["TVT"].values.astype(np.float64)
    t_gr  = t["GR"].values.astype(np.float64)

    gr_s = pd.Series(t_gr)
    gr_roll10 = gr_s.rolling(10, center=True, min_periods=1).mean().values
    gr_roll50 = gr_s.rolling(50, center=True, min_periods=1).mean().values
    dgr = np.gradient(t_gr, t_tvt, edge_order=1)
    dgr = np.nan_to_num(dgr, 0.0)

    t_feat = np.stack([t_gr, gr_roll10, gr_roll50, dgr], axis=1)

    diffs = np.abs(np.diff(t_tvt))
    diffs = diffs[diffs > 0]
    step = np.median(diffs) if len(diffs) > 0 else 0.5
    ratio = step / target_step

    if np.isclose(ratio, 1.0):
        pass
    elif ratio < 1.0:
        group_size = int(round(1 / ratio))
        n = len(t_tvt)
        pad_len = (-n) % group_size
        if pad_len > 0:
            t_tvt = np.pad(t_tvt, (0, pad_len), mode="edge")
            t_feat = np.pad(t_feat, ((0, pad_len), (0, 0)), mode="edge")
        t_tvt = t_tvt.reshape(-1, group_size).mean(axis=1)
        t_feat = t_feat.reshape(-1, group_size, t_feat.shape[1]).mean(axis=1)
    else:
        up_factor = int(round(ratio))
        old_idx = np.arange(len(t_tvt))
        new_idx = np.linspace(0, len(t_tvt) - 1, (len(t_tvt) - 1) * up_factor + 1)
        new_tvt = np.interp(new_idx, old_idx, t_tvt)
        new_feat = np.stack([np.interp(new_idx, old_idx, t_feat[:, k]) for k in range(t_feat.shape[1])], axis=1)
        t_tvt = new_tvt
        t_feat = new_feat

    return t_tvt, t_feat


def resample_horizontal_by_step(h, target_step=32, offset=0):
    h = h.copy()
    
    h_gr_raw = h["GR"].astype(float).interpolate().bfill().ffill().fillna(85.0).values
    if len(h_gr_raw) > Config.H_GR_FILTER:
        h_gr_smooth = savgol_filter(h_gr_raw, Config.H_GR_FILTER, 2)
    else:
        h_gr_smooth = h_gr_raw.copy()
    
    gr_s = pd.Series(h_gr_smooth)
    gr_roll10 = gr_s.rolling(10, center=True, min_periods=1).mean().values
    gr_roll50 = gr_s.rolling(50, center=True, min_periods=1).mean().values
    dgr = np.gradient(h_gr_smooth)
    dgr = np.nan_to_num(dgr, 0.0)

    Z = h["Z"].values.astype(float) if "Z" in h.columns else np.zeros(len(h))
    MD = h["MD"].values.astype(float) if "MD" in h.columns else np.arange(len(h), dtype=float)
    X = h["X"].values.astype(float) if "X" in h.columns else np.zeros(len(h))
    Y = h["Y"].values.astype(float) if "Y" in h.columns else np.zeros(len(h))
    
    dz = np.gradient(Z); dz = np.nan_to_num(dz, 0.0)
    dmd = np.gradient(MD); dmd = np.nan_to_num(dmd, 1.0)
    dz_dmd = dz / (np.abs(dmd) + 1e-8)
    dx = np.gradient(X); dx = np.nan_to_num(dx, 0.0)
    dy = np.gradient(Y); dy = np.nan_to_num(dy, 0.0)
    azimuth = np.arctan2(dy, dx + 1e-8)
    relative_md = MD - MD.min()
    relative_md = relative_md / (relative_md.max() + 1e-8)
    
    tvt_input = h["TVT_input"].values.astype(float) if "TVT_input" in h.columns else np.full(len(h), np.nan)
    c_est = tvt_input + Z
    c_est = pd.Series(c_est).interpolate(limit_direction='both').ffill().bfill().fillna(0.0).values
    c_est = (c_est - c_est.mean()) / (c_est.std() + 1e-8)

    # FIXED Fallback logic for test set evaluation
    if "TVT" in h.columns and h["TVT"].notna().sum() > 0:
        tvt_vals = h["TVT"].values.astype(float)
    elif "TVT_input" in h.columns and h["TVT_input"].notna().sum() > 0:
        tvt_vals = h["TVT_input"].astype(float).ffill().bfill().fillna(0.0).values
    else:
        tvt_vals = h["Z"].values.astype(float)
    
    data = np.stack([
        h_gr_smooth,   # 0: GR_smooth
        dz,            # 1: dZ
        dmd,           # 2: dMD
        dz_dmd,        # 3: dZ/dMD
        dx,            # 4: dX
        dy,            # 5: dY
        azimuth,       # 6: azimuth
        gr_roll10,     # 7: GR_roll10
        gr_roll50,     # 8: GR_roll50
        dgr,           # 9: dGR
        relative_md,   # 10: relative_md
        c_est,         # 11: C_estimate
    ], axis=1)

    if "TVT_input" in h.columns and h["TVT_input"].notna().sum() > 0:
        h_ps = int(np.flatnonzero(h["TVT_input"].notna().values)[-1]) + offset
    else:
        h_ps = len(h) // 2

    tvt_all = tvt_vals
    before_tvt = tvt_all[:h_ps+1]
    after_tvt  = tvt_all[h_ps+1:]
    before_feat = data[:h_ps+1]
    after_feat  = data[h_ps+1:]

    def bin_avg(arr, step):
        if len(arr) == 0:
            return np.array([])
        pad_n = (-len(arr)) % step
        if pad_n < step // 2 and len(arr) > 0:
            arr = np.pad(arr, ((pad_n, 0),) + ((0, 0),) * (arr.ndim - 1), mode="edge")
        elif len(arr) > 0 and pad_n > 0:
            arr = arr[(step - pad_n):]
        if len(arr) == 0:
            return np.array([])
        new_shape = (len(arr) // step, step) + arr.shape[1:]
        return arr.reshape(new_shape).mean(axis=1)

    def bin_avg_1d(arr, step):
        if len(arr) == 0:
            return np.array([])
        pad_n = (-len(arr)) % step
        if pad_n < step // 2:
            arr = np.pad(arr, (pad_n, 0), mode="edge")
        elif pad_n > 0:
            arr = arr[(step - pad_n):]
        if len(arr) == 0:
            return np.array([])
        return arr.reshape(-1, step).mean(axis=1)

    def bin_avg_after(arr, step):
        if len(arr) == 0:
            return np.array([])
        pad_n = (-len(arr)) % step
        if pad_n < step // 2 and len(arr) > 0:
            arr = np.pad(arr, ((0, pad_n),) + ((0, 0),) * (arr.ndim - 1), mode="edge")
        elif len(arr) > 0 and pad_n > 0:
            arr = arr[:-(step - pad_n)]
        if len(arr) == 0:
            return np.array([])
        new_shape = (len(arr) // step, step) + arr.shape[1:]
        return arr.reshape(new_shape).mean(axis=1)

    def bin_avg_after_1d(arr, step):
        if len(arr) == 0:
            return np.array([])
        pad_n = (-len(arr)) % step
        if pad_n < step // 2:
            arr = np.pad(arr, (0, pad_n), mode="edge")
        elif pad_n > 0:
            arr = arr[:-(step - pad_n)]
        if len(arr) == 0:
            return np.array([])
        return arr.reshape(-1, step).mean(axis=1)

    h_tvt0  = bin_avg_1d(before_tvt, target_step)
    h_feat0 = bin_avg(before_feat, target_step)
    h_tvt1  = bin_avg_after_1d(after_tvt, target_step)
    h_feat1 = bin_avg_after(after_feat, target_step)
    
    K_H = 12
    if h_feat0.ndim != 2: h_feat0 = np.zeros((0, K_H))
    if h_feat1.ndim != 2: h_feat1 = np.zeros((0, K_H))

    return h_tvt0, h_tvt1, h_feat0, h_feat1


def get_crop_index_and_pad_1d(n, center, history, future):
    raw_i0 = center - history
    raw_i1 = center + future
    i0 = max(raw_i0, 0)
    i1 = min(raw_i1, n)
    pad_left = max(0, -int(raw_i0))
    pad_right = max(0, int(raw_i1 - n))
    return i0, i1, pad_left, pad_right


class WellboreSDFDataset(Dataset):
    K_T = 4
    K_H = 12

    def __init__(self, well_files, config=Config, is_train=True, offset=0, h_step=64):
        self.well_files = well_files
        self.config = config
        self.is_train = is_train
        self.offset = offset
        self.h_step = h_step
        self.H_S = self.h_step // self.config.NN_SCALE

    def __len__(self):
        return len(self.well_files)

    def __getitem__(self, idx):
        horiz_path = self.well_files[idx]
        sample_id = horiz_path.name.split("__")[0]

        h = pd.read_csv(horiz_path)
        typewell_path = horiz_path.parent / f"{sample_id}{self.config.TYPEWELL_SUFFIX}"
        t = pd.read_csv(typewell_path)

        t_tvt, t_feat = resample_typewell_by_step(t, target_step=0.5)
        h_tvt0, h_tvt1, h_feat0, h_feat1 = resample_horizontal_by_step(
            h, target_step=self.H_S, offset=self.offset)

        if len(h_tvt0) > 0:
            last_tvt = h_tvt0[-1]
            last_idx = np.abs(t_tvt - last_tvt).argmin()
        else:
            last_idx = len(t_tvt) // 2

        j0, j1, pl, pr = get_crop_index_and_pad_1d(
            len(t_tvt), last_idx + 1, history=self.config.T_H, future=self.config.T_F)

        t_seg_mask = np.pad(np.ones(j1 - j0), (pl, pr))
        t_seg_tvt  = np.pad(t_tvt[j0:j1], (pl, pr), mode="edge")
        t_seg_feat = np.pad(t_feat[j0:j1], ((pl, pr), (0, 0)), mode="edge") if len(t_feat) > 0 \
            else np.zeros((self.config.T_H + self.config.T_F, self.K_T))
        t_seg_gr = t_seg_feat[:, 0]

        j0_h0, j1_h0, pl_h0, pr_h0 = get_crop_index_and_pad_1d(
            len(h_tvt0), len(h_tvt0), history=self.config.H_H, future=0)

        h_seg_mask0 = np.pad(np.ones(j1_h0 - j0_h0), (pl_h0, pr_h0))
        h_seg_tvt0  = np.pad(h_tvt0[j0_h0:j1_h0], (pl_h0, pr_h0), mode="edge") \
            if len(h_tvt0) > 0 else np.zeros(self.config.H_H)
        h_seg_feat0 = np.pad(h_feat0[j0_h0:j1_h0], ((pl_h0, pr_h0), (0, 0)), mode="edge") \
            if len(h_feat0) > 0 else np.zeros((self.config.H_H, self.K_H))

        j0_h1, j1_h1, pl_h1, pr_h1 = get_crop_index_and_pad_1d(
            len(h_tvt1), 0, history=0, future=self.config.H_F)

        h_seg_mask1 = np.pad(np.ones(j1_h1 - j0_h1), (pl_h1, pr_h1))
        h_seg_tvt1  = np.pad(h_tvt1[j0_h1:j1_h1], (pl_h1, pr_h1), mode="edge") \
            if len(h_tvt1) > 0 else np.zeros(self.config.H_F)
        h_seg_feat1 = np.pad(h_feat1[j0_h1:j1_h1], ((pl_h1, pr_h1), (0, 0)), mode="edge") \
            if len(h_feat1) > 0 else np.zeros((self.config.H_F, self.K_H))

        h_seg_mask = np.concatenate([h_seg_mask0, h_seg_mask1])
        h_seg_tvt  = np.concatenate([h_seg_tvt0, h_seg_tvt1])
        h_seg_feat = np.concatenate([h_seg_feat0, h_seg_feat1], axis=0)
        h_seg_gr   = h_seg_feat[:, 0]

        history_mask = np.zeros(self.config.H_H + self.config.H_F, dtype=np.float32)
        true_h0_len = j1_h0 - j0_h0
        if true_h0_len > 0:
            history_mask[pl_h0: pl_h0 + true_h0_len] = 1.0

        eval_mask_h = np.zeros(self.config.H_H + self.config.H_F, dtype=np.float32)
        true_h1_len = j1_h1 - j0_h1
        if true_h1_len > 0:
            eval_mask_h[self.config.H_H + pl_h1: self.config.H_H + pl_h1 + true_h1_len] = 1.0

        sdf = (h_seg_tvt[None, :] - t_seg_tvt[:, None]) / 40.0
        sdf = np.clip(sdf, -3, 3)

        diff = np.abs(t_seg_tvt[:, None] - h_seg_tvt[None, :])
        matched = diff.argmin(0)
        matched_mask = (diff.min(0) < 1).astype(np.float32) * h_seg_mask

        H = len(h_seg_gr)
        T = len(t_seg_gr)
        label = np.zeros((T, H), dtype=np.float32)
        for i in range(H - 1):
            if matched_mask[i] == 0 or matched_mask[i + 1] == 0:
                continue
            cv2.line(label, (i, matched[i]), (i + 1, matched[i + 1]), 1.0, 6, cv2.LINE_AA)

        history = label.copy()
        history[:, self.config.H_H + 1:] = 0
        target = matched

        # Horizontal Flip Data Augmentation (Only during training to avoid overfitting)
        if self.is_train and np.random.rand() < 0.5:
            h_seg_mask = h_seg_mask[::-1].copy()
            h_seg_tvt = h_seg_tvt[::-1].copy()
            h_seg_feat = h_seg_feat[::-1, :].copy()
            h_seg_gr = h_seg_gr[::-1].copy()
            history_mask = history_mask[::-1].copy()
            eval_mask_h = eval_mask_h[::-1].copy()
            sdf = sdf[:, ::-1].copy()
            label = label[:, ::-1].copy()
            history = history[:, ::-1].copy()
            target = target[::-1].copy()
            matched = matched[::-1].copy()
            matched_mask = matched_mask[::-1].copy()

        orig_tvt = h["TVT"].values if ("TVT" in h.columns and h["TVT"].notna().sum() > 0) \
            else np.zeros(len(h))
        orig_len = len(orig_tvt)
        padded_orig = np.zeros(self.config.ORIG_PAD_LEN, dtype=np.float32)
        use_len = min(orig_len, self.config.ORIG_PAD_LEN)
        padded_orig[:use_len] = orig_tvt[:use_len]

        if "TVT_input" in h.columns and h["TVT_input"].notna().sum() > 0:
            h_ps = int(np.flatnonzero(h["TVT_input"].notna().values)[-1]) + self.offset
        else:
            h_ps = len(h) // 2

        # The math anchor index is mathematically constant: self.config.T_H - 1
        anchor_t_idx = self.config.T_H - 1

        return {
            "id": sample_id,
            "t_gr":   torch.tensor(t_seg_gr, dtype=torch.float32),
            "h_gr":   torch.tensor(h_seg_gr, dtype=torch.float32),
            "t_feat": torch.tensor(t_seg_feat, dtype=torch.float32).T,
            "h_feat": torch.tensor(h_seg_feat, dtype=torch.float32).T,
            "history":      torch.tensor(history, dtype=torch.float32).unsqueeze(0),
            "label":        torch.tensor(label, dtype=torch.float32).unsqueeze(0),
            "target":       torch.tensor(target, dtype=torch.int64),
            "h_mask":       torch.tensor(h_seg_mask, dtype=torch.float32),
            "t_mask":       torch.tensor(t_seg_mask, dtype=torch.float32),
            "sdf":          torch.tensor(sdf, dtype=torch.float32).unsqueeze(0),
            "eval_mask":    torch.tensor(eval_mask_h, dtype=torch.float32),
            "t_seg_tvt":    torch.tensor(t_seg_tvt, dtype=torch.float32),
            "h_seg_tvt":    torch.tensor(h_seg_tvt, dtype=torch.float32),
            "history_mask": torch.tensor(history_mask, dtype=torch.float32),
            "matched_idx":  torch.tensor(matched, dtype=torch.int64),
            "orig_tvt":     torch.tensor(padded_orig, dtype=torch.float32),
            "orig_len":     torch.tensor(orig_len, dtype=torch.int64),
            "h_ps":         torch.tensor(h_ps, dtype=torch.int64),
            "anchor_t_idx": torch.tensor(anchor_t_idx, dtype=torch.int64),
        }

In [ ]:
%%writefile model.py
# model.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, max(1, channel // reduction), bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(max(1, channel // reduction), channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class ResBlockWithSE(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.se = SEBlock(out_c)
        
        self.downsample = None
        if stride != 1 or in_c != out_c:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_c)
            )

    def forward(self, x):
        identity = x
        out = F.gelu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return F.gelu(out)

class UNetResNet(nn.Module):
    def __init__(self, in_c=21):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_c, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU()
        )
        
        self.layer1 = self._make_layer(32, 64, blocks=2, stride=2)
        self.layer2 = self._make_layer(64, 128, blocks=2, stride=2)
        self.layer3 = self._make_layer(128, 256, blocks=2, stride=2)
        self.layer4 = self._make_layer(256, 512, blocks=2, stride=2)

    def _make_layer(self, in_c, out_c, blocks, stride):
        layers = [ResBlockWithSE(in_c, out_c, stride)]
        for _ in range(1, blocks):
            layers.append(ResBlockWithSE(out_c, out_c, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        f0 = self.stem(x)
        f1 = self.layer1(f0)
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)
        return f0, f1, f2, f3, f4

class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU()
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=4, dilation=4, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU()
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=8, dilation=8, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU()
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=12, dilation=12, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU()
        )
        self.global_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU()
        )
        self.out_conv = nn.Sequential(
            nn.Conv2d(out_channels * 5, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        size = x.shape[-2:]
        x1 = self.conv1(x)
        x2 = self.conv2(x)
        x3 = self.conv3(x)
        x4 = self.conv4(x)
        x5 = self.global_pool(x)
        x5 = F.interpolate(x5, size=size, mode='bilinear', align_corners=False)
        out = torch.cat([x1, x2, x3, x4, x5], dim=1)
        return self.out_conv(out)

class UpBlock(nn.Module):
    def __init__(self, in_c, skip_c, out_c):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv = nn.Sequential(
            nn.Conv2d(in_c + skip_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.GELU(),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.GELU()
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class GeoSteerNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.output_type = ["inference", "loss"]
        self.D = nn.Parameter(torch.zeros(1))

        self.norm = nn.InstanceNorm2d(17)
        self.backbone = UNetResNet(in_c=21)

        self.aspp = ASPP(512, 256)
        
        self.up3 = UpBlock(256, 256, 128)
        self.up2 = UpBlock(128, 128, 64)
        self.up1 = UpBlock(64, 64, 32)
        self.up0 = UpBlock(32, 32, 32)

        self.head = nn.Sequential(
            nn.Conv2d(34, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.Conv2d(32, 1, kernel_size=1)
        )

    def forward(self, batch):
        device = self.D.device
        t_gr = batch["t_gr"].to(device)
        h_gr = batch["h_gr"].to(device)
        t_tvt = batch["t_seg_tvt"].to(device)
        h_tvt = batch["h_seg_tvt"].to(device)
        h_hist_mask = batch["history_mask"].to(device)
        t_feat = batch["t_feat"].to(device) 
        h_feat = batch["h_feat"].to(device) 
        
        t_mask = batch["t_mask"].to(device)
        h_mask = batch["h_mask"].to(device)

        B, T = t_gr.shape
        _, H = h_gr.shape
        
        t_gr_img = t_gr.reshape(B, 1, T, 1).expand(B, 1, T, H)
        h_gr_img = h_gr.reshape(B, 1, 1, H).expand(B, 1, T, H)
        gr_diff = t_gr_img - h_gr_img
        
        t_tvt_img = t_tvt.reshape(B, 1, T, 1).expand(B, 1, T, H)
        h_tvt_img = h_tvt.reshape(B, 1, 1, H).expand(B, 1, T, H)
        
        # FIXED Scale tvt_diff by 40.0 to physically normalize history_img to [-3, 3] range
        tvt_diff = (t_tvt_img - h_tvt_img) / 40.0
        
        hist_mask_img = h_hist_mask.reshape(B, 1, 1, H).expand(B, 1, T, H)
        history_img = tvt_diff * hist_mask_img
        
        t_extra = t_feat[:, 1:, :].reshape(B, 3, T, 1).expand(B, 3, T, H)
        h_extra = h_feat[:, 1:, :].reshape(B, 11, 1, H).expand(B, 11, T, H)

        continuous_feats = torch.cat([
            t_gr_img,
            h_gr_img,
            gr_diff,
            t_extra,
            h_extra
        ], dim=1) # 17 channels
        
        continuous_feats = self.norm(continuous_feats)

        image = torch.cat([
            continuous_feats,
            history_img,
            hist_mask_img
        ], dim=1) # 19 channels total
        
        t_coords = torch.linspace(-1, 1, T, dtype=torch.float32, device=device).view(1, 1, T, 1).expand(B, 1, T, H)
        h_coords = torch.linspace(-1, 1, H, dtype=torch.float32, device=device).view(1, 1, 1, H).expand(B, 1, T, H)
        image = torch.cat([image, t_coords, h_coords], dim=1)
        
        f0, f1, f2, f3, f4 = self.backbone(image)
        
        x = self.aspp(f4)
        x = self.up3(x, f3)
        x = self.up2(x, f2)
        x = self.up1(x, f1)
        x = self.up0(x, f0)
        
        x = torch.cat([x, history_img, hist_mask_img], dim=1)
        
        sdf = self.head(x)
        sdf = torch.tanh(sdf) * 3

        output = {}
        if "loss" in self.output_type and "sdf" in batch:
            target = batch["sdf"].to(device)
            mask   = t_mask[:, None, :, None] * h_mask[:, None, None, :]
            
            sdf_loss = do_masked_hybrid_loss(sdf, target, mask)
            output["sdf_loss"] = sdf_loss

        if "inference" in self.output_type:
            output["sdf"] = sdf

        return output

def do_masked_hybrid_loss(predict, target, mask):
    mse = F.mse_loss(predict, target, reduction="none")
    l1 = F.l1_loss(predict, target, reduction="none")
    base_loss = mse + 0.5 * l1

    target_weight = 1.0 + 2.0 * torch.exp(-torch.square(target))
    
    loss = ((base_loss * target_weight) * mask).sum() / ((mask * target_weight).sum() + 1e-8)
    return loss

In [ ]:
%%writefile train.py

# train.py
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from config import Config
from dataset import WellboreSDFDataset
from model import GeoSteerNet
from torch.optim.lr_scheduler import CosineAnnealingLR
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import GroupKFold
import hashlib

def viterbi_decode_future(cost_matrix, anchor_t_idx, H_H, transition_penalty=0.10):
    """
    Finds the optimal path specifically in the evaluation (future) zone,
    forcing the sequence to start exactly from the known anchor position
    at index H_H - 1.
    """
    T, H = cost_matrix.shape
    dp = np.full((T, H), 1e9, dtype=np.float32)
    pointers = np.zeros((T, H), dtype=np.int32)
    
    # Force anchor starting state at H_H - 1
    dp[anchor_t_idx, H_H - 1] = cost_matrix[anchor_t_idx, H_H - 1]
    
    t_idx = np.arange(T)
    abs_diff = np.abs(t_idx[:, None] - t_idx[None, :])
    
    # Forward Pass starting from history boundary
    for h in range(H_H, H):
        prev_dp = dp[:, h-1]
        
        # trans_costs[i, j] = prev_dp[j] + penalty * |i - j|
        trans_costs = prev_dp[None, :] + transition_penalty * abs_diff
        
        # Best previous state index j for each current state i
        min_costs_idx = np.argmin(trans_costs, axis=1)
        
        dp[:, h] = cost_matrix[:, h] + trans_costs[t_idx, min_costs_idx]
        pointers[:, h] = min_costs_idx
        
    # Backward Pass starting from end of horizontal segment
    path = np.zeros(H, dtype=np.int32)
    path[:H_H] = anchor_t_idx  # Safe fallback for history indexes
    
    path[-1] = np.argmin(dp[:, -1])
    for h in range(H-1, H_H - 1, -1):
        path[h-1] = pointers[path[h], h]
        
    return path

def map_h_to_original(pred_tvt_H, o_len, h_ps, H_S, H_H, H_F):
    centers, vals = [], []
    
    # History mapping
    for k in range(H_H):
        h_idx = H_H - 1 - k
        val = pred_tvt_H[h_idx]
        center_i = h_ps - k * H_S - (H_S - 1) / 2.0
        centers.append(center_i)
        vals.append(val)
        
    # Future mapping
    for k in range(H_F):
        h_idx = H_H + k
        val = pred_tvt_H[h_idx]
        center_i = h_ps + 1 + k * H_S + (H_S - 1) / 2.0
        centers.append(center_i)
        vals.append(val)
        
    centers = np.array(centers)
    vals = np.array(vals)
    
    sort_idx = np.argsort(centers)
    centers = centers[sort_idx]
    vals = vals[sort_idx]
    
    orig_indices = np.arange(o_len)
    return np.interp(orig_indices, centers, vals)

def train_one_epoch(model, dataloader, optimizer, scaler, device):
    model.train()
    total_loss = 0.0
    for batch in tqdm(dataloader, desc="Training"):
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            output = model(batch)
            loss = output["sdf_loss"]
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(dataloader)

@torch.no_grad()
def evaluate_model(model, dataloader, device):
    model.eval()
    total_se = 0.0
    total_points = 0
    total_loss = 0.0
    
    for batch in tqdm(dataloader, desc="Evaluating"):
        with torch.amp.autocast('cuda'):
            output = model(batch)
            sdf = output["sdf"]
            loss = output["sdf_loss"]
            
        total_loss += loss.item()
        
        B = sdf.size(0)
        sdf_np = sdf.squeeze(1).cpu().numpy()
        t_seg_tvt = batch["t_seg_tvt"].numpy() 
        h_seg_tvt = batch["h_seg_tvt"].numpy()
        orig_tvt = batch["orig_tvt"].numpy() 
        orig_len = batch["orig_len"].numpy() 
        h_ps = batch["h_ps"].numpy() 
        anchor_t_idx = batch["anchor_t_idx"].numpy()
        
        for b in range(B):
            cost_mat = np.abs(sdf_np[b])
            
            best_t_idx = viterbi_decode_future(
                cost_mat, 
                anchor_t_idx=int(anchor_t_idx[b]), 
                H_H=Config.H_H, 
                transition_penalty=0.10
            )
            pred_tvt_H = t_seg_tvt[b][best_t_idx] 
            
            pred_tvt_H_eval = pred_tvt_H.copy()
            pred_tvt_H_eval[:Config.H_H] = h_seg_tvt[b][:Config.H_H]
            
            o_len = int(orig_len[b])
            ps = int(h_ps[b])
            
            pred_resampled = map_h_to_original(
                pred_tvt_H_eval, o_len, ps, 
                Config.H_S, Config.H_H, Config.H_F
            )
            
            true_tvt = orig_tvt[b][:o_len]
            
            if ps < o_len - 1:
                eval_pred = pred_resampled[ps + 1:]
                eval_true = true_tvt[ps + 1:]
                
                total_se += np.sum((eval_pred - eval_true)**2)
                total_points += len(eval_true)
    
    val_loss = total_loss / len(dataloader)
    val_rmse = np.sqrt(total_se / total_points) if total_points > 0 else 0.0
    return val_loss, val_rmse

def get_typewell_hash(well_file, train_dir):
    well_name = well_file.name.split("__")[0]
    typewell_file = train_dir / f"{well_name}{Config.TYPEWELL_SUFFIX}"
    if typewell_file.exists():
        try:
            df = pd.read_csv(typewell_file)
            rounded_gr = np.round(df["GR"].values, 2).tobytes()
            return hashlib.md5(rounded_gr).hexdigest()
        except Exception:
            return well_name
    return well_name

def run_training():
    train_dir = Path(Config.TRAIN_DIR)
    well_files = sorted(list(train_dir.glob(f"*{Config.HORIZONTAL_SUFFIX}")))
    if not well_files:
        print("No data found in", Config.TRAIN_DIR)
        return
    
    typewell_groups = [get_typewell_hash(f, train_dir) for f in well_files]
    gkf = GroupKFold(n_splits=Config.N_FOLDS)
    
    overall_val_loss = []

    for fold, (train_idx, val_idx) in enumerate(gkf.split(well_files, groups=typewell_groups)):
        print(f"\n{'='*20} Fold {fold} {'='*20}")
        train_files = [well_files[i] for i in train_idx]
        val_files = [well_files[i] for i in val_idx]
        
        train_ds = WellboreSDFDataset(well_files=train_files, is_train=True)
        val_ds = WellboreSDFDataset(well_files=val_files, is_train=False) 
        
        train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
        
        model = GeoSteerNet().to(Config.DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR, weight_decay=1e-4)
        scheduler = CosineAnnealingLR(optimizer, T_max=Config.EPOCHS)
        scaler = torch.amp.GradScaler('cuda')
        
        best_loss = float('inf')
        
        for epoch in range(1, Config.EPOCHS + 1):
            loss = train_one_epoch(model, train_loader, optimizer, scaler, Config.DEVICE)
            val_loss, val_rmse = evaluate_model(model, val_loader, Config.DEVICE)
            scheduler.step()
            
            status = ""
            if val_rmse < best_loss:
                best_loss = val_rmse
                torch.save(model.state_dict(), f"fold_{fold}_best.pth")
                status = " (Best RMSE Saved!)"
                
            print(f"Epoch {epoch:02d} | Train Loss: {loss:.4f} | Val Loss (SDF): {val_loss:.4f} | Val RMSE (ft): {val_rmse:.2f}{status}")
            
        overall_val_loss.append(best_loss)
        print(f"Fold {fold} Best Val RMSE: {best_loss:.2f}")

    print(f"\n{'='*20} Final Results {'='*20}")
    print(f"Average CV RMSE (ft): {np.mean(overall_val_loss):.2f}")

if __name__ == "__main__":
    run_training()

In [ ]:
!python train.py

In [ ]:
# infer.py
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader
from tqdm import tqdm

from config import Config
from dataset import WellboreSDFDataset, resample_horizontal_by_step, resample_typewell_by_step, get_crop_index_and_pad_1d
from model import GeoSteerNet


def viterbi_decode_future(cost_matrix, anchor_t_idx, H_H, transition_penalty=0.10):
    """
    Finds the optimal path specifically in the evaluation (future) zone,
    forcing the sequence to start exactly from the known anchor position
    at index H_H - 1.
    """
    T, H = cost_matrix.shape
    dp = np.full((T, H), 1e9, dtype=np.float32)
    pointers = np.zeros((T, H), dtype=np.int32)
    
    # Force anchor starting state at H_H - 1
    dp[anchor_t_idx, H_H - 1] = cost_matrix[anchor_t_idx, H_H - 1]
    
    t_idx = np.arange(T)
    abs_diff = np.abs(t_idx[:, None] - t_idx[None, :])
    
    # Forward Pass starting from history boundary
    for h in range(H_H, H):
        prev_dp = dp[:, h-1]
        
        # trans_costs[i, j] = prev_dp[j] + penalty * |i - j|
        trans_costs = prev_dp[None, :] + transition_penalty * abs_diff
        
        # Best previous state index j for each current state i
        min_costs_idx = np.argmin(trans_costs, axis=1)
        
        dp[:, h] = cost_matrix[:, h] + trans_costs[t_idx, min_costs_idx]
        pointers[:, h] = min_costs_idx
        
    # Backward Pass starting from end of horizontal segment
    path = np.zeros(H, dtype=np.int32)
    path[:H_H] = anchor_t_idx  # Safe fallback for history indexes
    
    path[-1] = np.argmin(dp[:, -1])
    for h in range(H-1, H_H - 1, -1):
        path[h-1] = pointers[path[h], h]
        
    return path


def map_h_to_original(pred_tvt_H, o_len, h_ps, H_S, H_H, H_F):
    """
    Maps the padded H-bins back to their exact original indices
    relative to the anchor split point (h_ps).
    """
    centers, vals = [], []
    
    # History mapping
    for k in range(H_H):
        h_idx = H_H - 1 - k
        val = pred_tvt_H[h_idx]
        center_i = h_ps - k * H_S - (H_S - 1) / 2.0
        centers.append(center_i)
        vals.append(val)
        
    # Future mapping
    for k in range(H_F):
        h_idx = H_H + k
        val = pred_tvt_H[h_idx]
        center_i = h_ps + 1 + k * H_S + (H_S - 1) / 2.0
        centers.append(center_i)
        vals.append(val)
        
    centers = np.array(centers)
    vals = np.array(vals)
    
    sort_idx = np.argsort(centers)
    centers = centers[sort_idx]
    vals = vals[sort_idx]
    
    orig_indices = np.arange(o_len)
    return np.interp(orig_indices, centers, vals)


def run_inference():
    test_dir = Path(Config.TEST_DIR)
    well_files = sorted(list(test_dir.glob(f"*{Config.HORIZONTAL_SUFFIX}")))
    if not well_files:
        print("No test data found.")
        return

    models = []
    for fold in range(Config.N_FOLDS):
        model_path = f"fold_{fold}_best.pth"
        if not Path(model_path).exists():
            continue
        model = GeoSteerNet().to(Config.DEVICE)
        model.output_type = ["inference"] # We need SDF only
        model.load_state_dict(torch.load(model_path, map_location=Config.DEVICE))
        model.eval()
        models.append(model)
        print(f"Loaded {model_path}")

    if not models:
        print("No models loaded. Exiting.")
        return
    print(f"Loaded {len(models)} models for ensemble.\n")

    submission_data = []

    for well_file in tqdm(well_files, desc="Inference"):
        well_name = well_file.name.split("__")[0]
        df = pd.read_csv(well_file)

        typewell_path = well_file.parent / f"{well_name}{Config.TYPEWELL_SUFFIX}"
        if not typewell_path.exists():
            print(f"Typewell missing for {well_name}. Skipping.")
            continue

        dataset = WellboreSDFDataset(well_files=[well_file], is_train=False)
        loader = DataLoader(dataset, batch_size=1, shuffle=False)

        # We assume batch size 1 exactly for inference loops
        for batch in loader:
            with torch.no_grad():
                fold_sdfs = []
                for model in models:
                    with torch.amp.autocast('cuda'):
                        output = model(batch)
                    # output["sdf"] shape: [B, 1, T, H]
                    fold_sdfs.append(output["sdf"].cpu().numpy())

                # Average SDF natively across folds
                avg_sdf = np.mean(fold_sdfs, axis=0) # [1, 1, T, H]
                avg_sdf = avg_sdf[0, 0] # [T, H]

                # Map typewell indices
                h = df
                t = pd.read_csv(typewell_path)
                t_tvt, t_feat = resample_typewell_by_step(t, target_step=0.5)
                h_tvt0, h_tvt1, h_feat0, h_feat1 = resample_horizontal_by_step(h, target_step=dataset.H_S, offset=0)

                last_tvt = h_tvt0[-1] if len(h_tvt0) > 0 else t_tvt[len(t_tvt)//2]
                last_idx = np.abs(t_tvt - last_tvt).argmin()
                j0, j1, pad_left, pad_right = get_crop_index_and_pad_1d(
                    len(t_tvt), last_idx+1, history=Config.T_H, future=Config.T_F)
                
                t_seg_tvt = np.pad(t_tvt[j0:j1], (pad_left, pad_right), mode="edge")

                # FIXED Derive sequence path using anchor-constrained Viterbi decoding
                anchor_t_idx = Config.T_H - 1
                best_t_idx = viterbi_decode_future(
                    np.abs(avg_sdf), 
                    anchor_t_idx=anchor_t_idx, 
                    H_H=Config.H_H, 
                    transition_penalty=0.10
                )

                # Pred TVT per H step
                pred_tvt_H = t_seg_tvt[best_t_idx]

                # FIXED Overwrite history portion with known resampled true history values
                h_seg_tvt = batch["h_seg_tvt"].numpy()[0]
                pred_tvt_H_eval = pred_tvt_H.copy()
                pred_tvt_H_eval[:Config.H_H] = h_seg_tvt[:Config.H_H]

                # Project H prediction space back to original sequence length
                if "TVT_input" in df.columns and df["TVT_input"].notna().sum() > 0:
                    h_ps = int(np.flatnonzero(df["TVT_input"].notna().values)[-1])
                else: 
                    h_ps = len(df) // 2

                # FIXED Exact coordinate mapping (prevents coordinate shifting)
                pred_resampled = map_h_to_original(
                    pred_tvt_H_eval, 
                    o_len=len(df), 
                    h_ps=h_ps, 
                    H_S=dataset.H_S, 
                    H_H=Config.H_H, 
                    H_F=Config.H_F
                )

                # Overwrite NaN indices
                final_tvt = df["TVT_input"].values.copy().astype(np.float64)
                if "TVT_input" not in df.columns:
                    final_tvt = np.full(len(df), np.nan)

                # Use model predictions for NaN
                for idx in range(len(df)):
                    if np.isnan(final_tvt[idx]):
                        final_tvt[idx] = pred_resampled[idx]

                # Only submit evaluation zone rows
                for idx in range(len(df)):
                    if np.isnan(df.iloc[idx]["TVT_input"] if "TVT_input" in df.columns else np.nan):
                        submission_data.append({
                            "id": f"{well_name}_{idx}",
                            "tvt": float(final_tvt[idx])
                        })

    sub_df = pd.DataFrame(submission_data)
    sub_df.to_csv("submission.csv", index=False)
    print(sub_df)
    print(f"\nDone: {len(sub_df)} rows")
    print("Submission saved to submission.csv")


if __name__ == "__main__":
    run_inference()